In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))


import cococo.layouts as layouts
import cococo.utils_routing as utils
import cococo.circuit_construction as circuit_construction
import cococo.internal_testing as internal_testing
import plotting

In [2]:
def print_vdp_layers_with_qubit_labels(vdp_layers, layout):
    """
    Print vdp_layers layer by layer.

    Converts gate keys from positions back to logical qubit labels using layout.

    Args:
        vdp_layers:
            list of dictionaries, each dictionary maps gate -> path.
            CNOT gate key: ((x1, y1), (x2, y2))
            T gate key: (x, y)

        layout:
            dict like {qubit_label: position}
    """
    layout_rev = {pos: q for q, pos in layout.items()}

    def gate_to_label(gate):
        # CNOT gate: ((x1, y1), (x2, y2))
        if isinstance(gate[0], tuple):
            q1 = layout_rev.get(gate[0], gate[0])
            q2 = layout_rev.get(gate[1], gate[1])
            return (q1, q2)

        # T gate / single-qubit gate: (x, y)
        if isinstance(gate[0], int):
            return layout_rev.get(gate, gate)

        return gate

    for layer_idx, layer in enumerate(vdp_layers):
        print(f"\nLayer {layer_idx}")
        print("-" * 40)

        if not layer:
            print("  empty layer")
            continue

        for gate, path in layer.items():
            gate_label = gate_to_label(gate)
            print(f"  gate {gate_label}:")
            print(f"    position key = {gate}")
            print(f"    path         = {path}")


## Example for Router with Movable Qubits

### Choose Layout

In [3]:
layout_type = "triple"
m = 4
n = 4
factories = []
remove_edges = False
g, data_qubit_locs, factory_ring = layouts.gen_layout_scalable(layout_type, m, n, factories, remove_edges)
layout = {i: j for i,j in enumerate(data_qubit_locs)} #mapping
t=2 #mock because we have only cnots here in the example

### Choose Circuit

In [4]:
layout_type = "triple"
m = 4
n = 4
factories = []
remove_edges = False
g, data_qubit_locs, factory_ring = layouts.gen_layout_scalable(layout_type, m, n, factories, remove_edges)
layout = {i: j for i,j in enumerate(data_qubit_locs)}
t=2

print("data qubit location", data_qubit_locs)
q = len(data_qubit_locs)
print("number of data qubits: ", q)
j = 8
num_gates = 50

data qubit location [(0, 0), (1, 0), (2, 0), (5, 0), (6, 0), (7, 0), (10, 0), (11, 0), (12, 0), (15, 0), (16, 0), (17, 0), (20, 0), (21, 0), (22, 0), (25, 0), (26, 0), (27, 0), (30, 0), (31, 0), (32, 0), (35, 0), (36, 0), (37, 0), (1, 2), (2, 2), (3, 2), (6, 2), (7, 2), (8, 2), (11, 2), (12, 2), (13, 2), (16, 2), (17, 2), (18, 2), (21, 2), (22, 2), (23, 2), (26, 2), (27, 2), (28, 2), (31, 2), (32, 2), (33, 2), (36, 2), (37, 2), (38, 2), (2, 4), (3, 4), (4, 4), (7, 4), (8, 4), (9, 4), (12, 4), (13, 4), (14, 4), (17, 4), (18, 4), (19, 4), (22, 4), (23, 4), (24, 4), (27, 4), (28, 4), (29, 4), (32, 4), (33, 4), (34, 4), (37, 4), (38, 4), (39, 4), (3, 6), (4, 6), (5, 6), (8, 6), (9, 6), (10, 6), (13, 6), (14, 6), (15, 6), (18, 6), (19, 6), (20, 6), (23, 6), (24, 6), (25, 6), (28, 6), (29, 6), (30, 6), (33, 6), (34, 6), (35, 6), (38, 6), (39, 6), (40, 6)]
number of data qubits:  96


### Run Standard Router with Standard Layout

In [5]:
dag, pairs = circuit_construction.create_random_sequential_circuit_dag(j, q, num_gates, ) # at least num_gates gates
print("number of gates: ", len(pairs))
terminal_pairs = layouts.translate_layout_circuit(pairs, layout) #let's stick to the simple layout
print("terminal pairs: ", terminal_pairs)

number of gates:  56
terminal pairs:  [((17, 2), (9, 4)), ((24, 4), (13, 2)), ((16, 0), (23, 2)), ((32, 2), (2, 0)), ((15, 0), (23, 4)), ((1, 0), (22, 0)), ((21, 2), (25, 0)), ((26, 2), (40, 6)), ((17, 2), (13, 2)), ((24, 4), (40, 6)), ((16, 0), (9, 4)), ((2, 0), (21, 2)), ((23, 2), (15, 0)), ((23, 4), (1, 0)), ((25, 0), (32, 2)), ((22, 0), (26, 2)), ((13, 2), (17, 2)), ((16, 0), (40, 6)), ((9, 4), (26, 2)), ((2, 0), (21, 2)), ((24, 4), (23, 2)), ((1, 0), (23, 4)), ((32, 2), (25, 0)), ((22, 0), (15, 0)), ((13, 2), (17, 2)), ((9, 4), (16, 0)), ((26, 2), (23, 2)), ((32, 2), (2, 0)), ((21, 2), (24, 4)), ((1, 0), (40, 6)), ((25, 0), (22, 0)), ((23, 4), (15, 0)), ((13, 2), (1, 0)), ((26, 2), (9, 4)), ((23, 2), (24, 4)), ((15, 0), (32, 2)), ((21, 2), (2, 0)), ((40, 6), (17, 2)), ((22, 0), (16, 0)), ((23, 4), (25, 0)), ((1, 0), (23, 4)), ((26, 2), (32, 2)), ((24, 4), (21, 2)), ((22, 0), (15, 0)), ((23, 2), (2, 0)), ((17, 2), (9, 4)), ((40, 6), (16, 0)), ((13, 2), (25, 0)), ((2, 0), (23, 4)), 

In [6]:
router = utils.BasicRouter(g, data_qubit_locs, factories, valid_path = "cc", t=t, metric = "exact", use_dag = True)
# each layer has disjoint logical support, however it doesn't guarantee that all those gates can be physically routed at the same time on the lattice
layers = router.split_layer_terminal_pairs(terminal_pairs)
vdp_layers, _ = router.find_total_vdp_layers_dyn(layers, data_qubit_locs, router.factory_times, layout, testing = True)
print("Len of schedule without teleportation: ", len(vdp_layers))
print_vdp_layers_with_qubit_labels(vdp_layers, layout)


2026-05-21 15:20:57,149 [INFO] root: stim test succeeded for standard routing (:
2026-05-21 15:20:57,150 [INFO] root: no duplicates found in standard routing (:
2026-05-21 15:20:57,152 [INFO] root: paths do not occupy logical pos (:
2026-05-21 15:20:57,152 [INFO] root: Reset times make sense, all good(:
Len of schedule without teleportation:  17

Layer 0
----------------------------------------
  gate (36, 15):
    position key = ((21, 2), (25, 0))
    path         = [(21, 2), (21, 1), (22, 1), (23, 1), (24, 1), (24, 0), (25, 0)]
  gate (34, 53):
    position key = ((17, 2), (9, 4))
    path         = [(17, 2), (17, 1), (16, 1), (15, 1), (15, 2), (14, 2), (14, 3), (13, 3), (12, 3), (11, 3), (10, 3), (9, 3), (9, 4)]
  gate (9, 61):
    position key = ((15, 0), (23, 4))
    path         = [(15, 0), (15, -1), (16, -1), (17, -1), (18, -1), (19, -1), (19, 0), (18, 0), (18, 1), (19, 1), (19, 2), (20, 2), (20, 3), (21, 3), (22, 3), (23, 3), (23, 4)]
  gate (39, 95):
    position key = ((26, 2

### Run Movable Qubits Router with Standard Initial Layout

In [7]:
router = utils.TeleportationRouter(g, data_qubit_locs, factories, valid_path="cc", t=t, metric="exact", use_dag = True, seed =  49218  )
layers = router.split_layer_terminal_pairs(terminal_pairs)

max_iters = 100
T_start = 100.0
T_end = 0.1
alpha = 0.95
t=4 #mock value for cnot circuit
radius = 10
k_lookahead = 5
metric = "exact"

steiner_init_type = "full_random"
jump_harvesting = True
stimtest = True

reduce_teleport = True
idle_move_type = "later"

schedule, _ = router.optimize_layers(        
        terminal_pairs,
        layout,
        max_iters,
        T_start,
        T_end,
        alpha,
        radius = radius,
        k_lookahead = k_lookahead,
        steiner_init_type = steiner_init_type,
        jump_harvesting = jump_harvesting,
        reduce_teleport = reduce_teleport,
        idle_move_type = idle_move_type,
        include_steiner_teleport = True,
        include_idle_teleport = True,
        reduce_init_steiner = True,
        reduce_init_idle = True, 
        stimtest = True, 
    )

2026-05-21 15:20:57,225 [INFO] root: Iteration 0: |vdp_dict|=6, pushing |terminal_pairs_remainder|=2, remaining |layers|=7
2026-05-21 15:21:26,764 [INFO] root: Steiner found for this layer.
2026-05-21 15:21:26,765 [INFO] root: Final Temperature T = 5.920529e-01
2026-05-21 15:21:29,687 [INFO] root: Complexity of teleportation could be reduced.
2026-05-21 15:21:29,921 [INFO] root: Iteration 1: |vdp_dict|=4, pushing |terminal_pairs_remainder|=2, remaining |layers|=3
2026-05-21 15:21:29,925 [INFO] root: No idling path back could be found for danger qubit (14, 1)
2026-05-21 15:21:29,928 [INFO] root: No idling path back could be found for danger qubit (12, -1)
2026-05-21 15:21:46,663 [INFO] root: Steiner found for this layer.
2026-05-21 15:21:46,663 [INFO] root: Final Temperature T = 5.920529e-01
2026-05-21 15:21:48,871 [INFO] root: Complexity of teleportation could be reduced.
2026-05-21 15:21:49,000 [INFO] root: Iteration 2: |vdp_dict|=5, pushing |terminal_pairs_remainder|=1, remaining |la

In [8]:
print("Len of schedule with teleport router: ", len(schedule))

Len of schedule with teleport router:  11


In [9]:
print("Reduction Delta: ", len(vdp_layers) - len(schedule))

Reduction Delta:  6


In [10]:
from IPython.display import HTML
from cococo.animation_routing_html import make_clean_routing_html_animation

anim = make_clean_routing_html_animation(
    g,
    schedule,
    initial_layout=layout,
    factories=factories,
    figsize=(18, 8),
    interval=900,
    save_path="routing_animation.html",
)

HTML(anim.to_jshtml())

2026-05-21 15:21:49,699 [INFO] matplotlib.animation: Animation.save using <class 'matplotlib.animation.HTMLWriter'>


In [16]:
print_vdp_layers_with_qubit_labels([schedule[0]["vdp_dict"]], schedule[0]["layout"])


Layer 0
----------------------------------------
  gate (36, 15):
    position key = ((21, 2), (25, 0))
    path         = [(21, 2), (21, 1), (22, 1), (23, 1), (24, 1), (24, 0), (25, 0)]
  gate (34, 53):
    position key = ((17, 2), (9, 4))
    path         = [(17, 2), (17, 1), (16, 1), (15, 1), (15, 2), (14, 2), (14, 3), (13, 3), (12, 3), (11, 3), (10, 3), (9, 3), (9, 4)]
  gate (9, 61):
    position key = ((15, 0), (23, 4))
    path         = [(15, 0), (15, -1), (16, -1), (17, -1), (18, -1), (19, -1), (19, 0), (18, 0), (18, 1), (19, 1), (19, 2), (20, 2), (20, 3), (21, 3), (22, 3), (23, 3), (23, 4)]
  gate (39, 95):
    position key = ((26, 2), (40, 6))
    path         = [(26, 2), (26, 3), (27, 3), (28, 3), (29, 3), (30, 3), (31, 3), (32, 3), (33, 3), (34, 3), (35, 3), (35, 4), (36, 4), (36, 5), (37, 5), (38, 5), (39, 5), (40, 5), (41, 5), (41, 6), (40, 6)]
  gate (62, 32):
    position key = ((24, 4), (13, 2))
    path         = [(24, 4), (24, 5), (23, 5), (22, 5), (21, 5), (20, 5)

In [18]:
print(schedule[0]["steiner"])

{((17, 2), (9, 4), (14, 1)): ([(17, 2), (17, 1), (16, 1), (15, 1), (15, 2), (14, 2), (14, 3), (13, 3), (12, 3), (11, 3), (10, 3), (9, 3), (9, 4)], [(15, 1), (14, 1)]), ((15, 0), (23, 4), (12, -1)): ([(15, 0), (15, -1), (16, -1), (17, -1), (18, -1), (19, -1), (19, 0), (18, 0), (18, 1), (19, 1), (19, 2), (20, 2), (20, 3), (21, 3), (22, 3), (23, 3), (23, 4)], [(15, -1), (14, -1), (13, -1), (12, -1)])}


In [19]:
print_vdp_layers_with_qubit_labels([schedule[1]["vdp_dict"]], schedule[1]["layout"])


Layer 0
----------------------------------------
  gate (34, 32):
    position key = ((14, 1), (13, 2))
    path         = [(14, 1), (13, 1), (13, 2)]
  gate (14, 39):
    position key = ((22, 0), (26, 2))
    path         = [(22, 0), (23, 0), (24, 0), (24, 1), (25, 1), (25, 2), (26, 2)]
  gate (10, 38):
    position key = ((16, 0), (23, 2))
    path         = [(16, 0), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (23, 2)]
  gate (61, 1):
    position key = ((12, -1), (1, 0))
    path         = [(12, -1), (11, -1), (10, -1), (9, -1), (8, -1), (7, -1), (6, -1), (5, -1), (4, -1), (3, -1), (2, -1), (1, -1), (1, 0)]
  gate (62, 95):
    position key = ((24, 4), (40, 6))
    path         = [(24, 4), (25, 4), (26, 4), (26, 5), (27, 5), (28, 5), (29, 5), (30, 5), (31, 5), (32, 5), (33, 5), (34, 5), (35, 5), (36, 5), (37, 5), (38, 5), (39, 5), (40, 5), (41, 5), (41, 6), (40, 6)]
  gate (43, 2):
    position key = ((32, 2), (2, 0))
    path         = [(32, 2), (32, 3

In [20]:
print_vdp_layers_with_qubit_labels([schedule[2]["vdp_dict"]], schedule[2]["layout"])


Layer 0
----------------------------------------
  gate (32, 34):
    position key = ((13, 2), (14, 1))
    path         = [(13, 2), (13, 1), (14, 1)]
  gate (10, 53):
    position key = ((16, 0), (9, 4))
    path         = [(16, 0), (16, 1), (15, 1), (15, 2), (14, 2), (14, 3), (13, 3), (12, 3), (11, 3), (10, 3), (9, 3), (9, 4)]
  gate (1, 61):
    position key = ((1, 0), (12, -1))
    path         = [(1, 0), (1, -1), (2, -1), (3, -1), (4, -1), (5, -1), (6, -1), (7, -1), (8, -1), (9, -1), (10, -1), (11, -1), (12, -1)]
  gate (15, 43):
    position key = ((25, 0), (32, 2))
    path         = [(25, 0), (24, 0), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (29, 2), (30, 2), (30, 3), (31, 3), (32, 3), (32, 2)]
  gate (38, 9):
    position key = ((23, 2), (15, 0))
    path         = [(23, 2), (23, 1), (22, 1), (21, 1), (20, 1), (19, 1), (18, 1), (18, 0), (19, 0), (19, -1), (18, -1), (17, -1), (16, -1), (15, -1), (15, 0)]
  gate (2, 36):
    position key = ((2, 0), (21, 2))
    pat

In [21]:
print_vdp_layers_with_qubit_labels([schedule[3]["vdp_dict"]], schedule[3]["layout"])


Layer 0
----------------------------------------
  gate (32, 34):
    position key = ((13, 2), (14, 1))
    path         = [(13, 2), (13, 1), (14, 1)]
  gate (62, 38):
    position key = ((24, 4), (23, 2))
    path         = [(24, 4), (23, 4), (23, 3), (24, 3), (24, 2), (23, 2)]
  gate (14, 9):
    position key = ((22, 0), (15, 0))
    path         = [(22, 0), (23, 0), (23, -1), (22, -1), (21, -1), (20, -1), (19, -1), (18, -1), (17, -1), (16, -1), (15, -1), (15, 0)]
  gate (43, 15):
    position key = ((32, 2), (25, 0))
    path         = [(32, 2), (32, 3), (31, 3), (30, 3), (30, 2), (29, 2), (29, 1), (28, 1), (27, 1), (26, 1), (25, 1), (24, 1), (24, 0), (25, 0)]
  gate (2, 36):
    position key = ((2, 0), (21, 2))
    path         = [(2, 0), (3, 0), (4, 0), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (9, 2), (10, 2), (10, 3), (11, 3), (12, 3), (13, 3), (14, 3), (15, 3), (16, 3), (17, 3), (18, 3), (19, 3), (20, 3), (20, 2), (21, 2)]
  gate (53, 39):
    position key = ((9, 4), (26

In [22]:
print_vdp_layers_with_qubit_labels([schedule[4]["vdp_dict"]], schedule[4]["layout"])


Layer 0
----------------------------------------
  gate (15, 14):
    position key = ((25, 0), (22, 0))
    path         = [(25, 0), (24, 0), (23, 0), (22, 0)]
  gate (39, 38):
    position key = ((26, 2), (23, 2))
    path         = [(26, 2), (25, 2), (24, 2), (23, 2)]
  gate (61, 9):
    position key = ((12, -1), (15, 0))
    path         = [(12, -1), (13, -1), (13, 0), (14, 0), (15, 0)]
  gate (36, 62):
    position key = ((21, 2), (24, 4))
    path         = [(21, 2), (20, 2), (20, 3), (21, 3), (22, 3), (23, 3), (23, 4), (24, 4)]
  gate (10, 95):
    position key = ((16, 0), (40, 6))
    path         = [(16, 0), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (29, 2), (30, 2), (30, 3), (31, 3), (32, 3), (33, 3), (34, 3), (35, 3), (35, 4), (36, 4), (36, 5), (37, 5), (38, 5), (39, 5), (40, 5), (41, 5), (41, 6), (40, 6)]


In [23]:
print_vdp_layers_with_qubit_labels([schedule[5]["vdp_dict"]], schedule[5]["layout"])


Layer 0
----------------------------------------
  gate (38, 62):
    position key = ((23, 2), (24, 4))
    path         = [(23, 2), (24, 2), (24, 3), (23, 3), (23, 4), (24, 4)]
  gate (53, 10):
    position key = ((9, 4), (16, 0))
    path         = [(9, 4), (9, 3), (10, 3), (11, 3), (12, 3), (13, 3), (14, 3), (14, 2), (15, 2), (15, 1), (16, 1), (16, 0)]
  gate (61, 15):
    position key = ((12, -1), (25, 0))
    path         = [(12, -1), (13, -1), (14, -1), (15, -1), (16, -1), (17, -1), (18, -1), (19, -1), (20, -1), (21, -1), (22, -1), (23, -1), (23, 0), (24, 0), (25, 0)]
  gate (43, 2):
    position key = ((32, 2), (2, 0))
    path         = [(32, 2), (32, 3), (31, 3), (31, 4), (30, 4), (30, 5), (29, 5), (28, 5), (27, 5), (26, 5), (25, 5), (24, 5), (23, 5), (22, 5), (21, 5), (20, 5), (19, 5), (18, 5), (17, 5), (16, 5), (15, 5), (14, 5), (13, 5), (12, 5), (11, 5), (10, 5), (9, 5), (8, 5), (7, 5), (6, 5), (6, 4), (5, 4), (5, 3), (4, 3), (4, 2), (5, 2), (5, 1), (4, 1), (4, 0), (3, 0),

In [24]:
print_vdp_layers_with_qubit_labels([schedule[6]["vdp_dict"]], schedule[6]["layout"])


Layer 0
----------------------------------------
  gate (14, 10):
    position key = ((22, 0), (16, 0))
    path         = [(22, 0), (22, 1), (21, 1), (20, 1), (19, 1), (18, 1), (17, 1), (16, 1), (16, 0)]
  gate (32, 1):
    position key = ((13, 2), (1, 0))
    path         = [(13, 2), (13, 1), (12, 1), (11, 1), (10, 1), (9, 1), (8, 1), (7, 1), (6, 1), (5, 1), (4, 1), (4, 0), (3, 0), (3, -1), (2, -1), (1, -1), (1, 0)]
  gate (39, 53):
    position key = ((26, 2), (9, 4))
    path         = [(26, 2), (26, 3), (25, 3), (24, 3), (23, 3), (22, 3), (21, 3), (20, 3), (19, 3), (18, 3), (17, 3), (16, 3), (15, 3), (14, 3), (13, 3), (12, 3), (11, 3), (10, 3), (9, 3), (9, 4)]
  gate (9, 43):
    position key = ((15, 0), (32, 2))
    path         = [(15, 0), (15, -1), (16, -1), (17, -1), (18, -1), (19, -1), (20, -1), (21, -1), (22, -1), (23, -1), (23, 0), (24, 0), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (29, 2), (30, 2), (30, 3), (31, 3), (32, 3), (32, 2)]


In [26]:
print(schedule[6]["steiner"])
print(schedule[6]["idle_teleport"])

{((13, 2), (1, 0), (32, -1)): ([(13, 2), (13, 1), (12, 1), (11, 1), (10, 1), (9, 1), (8, 1), (7, 1), (6, 1), (5, 1), (4, 1), (4, 0), (3, 0), (3, -1), (2, -1), (1, -1), (1, 0)], [(5, 1), (5, 2), (4, 2), (4, 3), (5, 3), (5, 4), (6, 4), (6, 5), (7, 5), (8, 5), (9, 5), (10, 5), (11, 5), (12, 5), (13, 5), (14, 5), (15, 5), (16, 5), (17, 5), (18, 5), (19, 5), (20, 5), (21, 5), (22, 5), (23, 5), (24, 5), (25, 5), (26, 5), (27, 5), (28, 5), (29, 5), (30, 5), (31, 5), (32, 5), (33, 5), (34, 5), (35, 5), (36, 5), (36, 4), (35, 4), (35, 3), (34, 3), (34, 2), (35, 2), (35, 1), (34, 1), (34, 0), (33, 0), (33, -1), (32, -1)])}
{('idle', (12, -1), (8, 0)): ([(12, -1), (11, -1), (10, -1), (9, -1), (9, 0), (8, 0)], None)}


In [27]:
print_vdp_layers_with_qubit_labels([schedule[7]["vdp_dict"]], schedule[7]["layout"])


Layer 0
----------------------------------------
  gate (32, 15):
    position key = ((32, -1), (25, 0))
    path         = [(32, -1), (31, -1), (30, -1), (29, -1), (28, -1), (27, -1), (26, -1), (25, -1), (25, 0)]
  gate (39, 43):
    position key = ((26, 2), (32, 2))
    path         = [(26, 2), (26, 3), (27, 3), (28, 3), (29, 3), (30, 3), (31, 3), (32, 3), (32, 2)]
  gate (14, 9):
    position key = ((22, 0), (15, 0))
    path         = [(22, 0), (23, 0), (23, -1), (22, -1), (21, -1), (20, -1), (19, -1), (18, -1), (17, -1), (16, -1), (15, -1), (15, 0)]
  gate (1, 61):
    position key = ((1, 0), (8, 0))
    path         = [(1, 0), (1, -1), (2, -1), (3, -1), (3, 0), (4, 0), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (8, 0)]
  gate (36, 2):
    position key = ((21, 2), (2, 0))
    path         = [(21, 2), (20, 2), (20, 3), (19, 3), (18, 3), (17, 3), (16, 3), (15, 3), (14, 3), (13, 3), (12, 3), (11, 3), (10, 3), (9, 3), (8, 3), (7, 3), (6, 3), (5, 3), (4, 3), (3, 3), (2, 3), (1, 3), (0, 3

In [28]:
print_vdp_layers_with_qubit_labels([schedule[8]["vdp_dict"]], schedule[8]["layout"])


Layer 0
----------------------------------------
  gate (62, 36):
    position key = ((24, 4), (21, 2))
    path         = [(24, 4), (23, 4), (23, 3), (22, 3), (21, 3), (20, 3), (20, 2), (21, 2)]
  gate (15, 32):
    position key = ((25, 0), (32, -1))
    path         = [(25, 0), (25, -1), (26, -1), (27, -1), (28, -1), (29, -1), (30, -1), (31, -1), (32, -1)]
  gate (34, 53):
    position key = ((14, 1), (9, 4))
    path         = [(14, 1), (13, 1), (12, 1), (11, 1), (10, 1), (9, 1), (9, 2), (10, 2), (10, 3), (9, 3), (9, 4)]
  gate (1, 14):
    position key = ((1, 0), (22, 0))
    path         = [(1, 0), (1, -1), (2, -1), (3, -1), (4, -1), (5, -1), (6, -1), (7, -1), (8, -1), (9, -1), (10, -1), (11, -1), (12, -1), (13, -1), (14, -1), (15, -1), (16, -1), (17, -1), (18, -1), (19, -1), (20, -1), (21, -1), (22, -1), (23, -1), (23, 0), (22, 0)]
  gate (95, 10):
    position key = ((40, 6), (16, 0))
    path         = [(40, 6), (41, 6), (41, 5), (40, 5), (39, 5), (38, 5), (37, 5), (36, 5), (3

In [29]:
print_vdp_layers_with_qubit_labels([schedule[9]["vdp_dict"]], schedule[9]["layout"])


Layer 0
----------------------------------------
  gate (39, 36):
    position key = ((26, 2), (21, 2))
    path         = [(26, 2), (25, 2), (25, 1), (24, 1), (23, 1), (22, 1), (21, 1), (21, 2)]
  gate (2, 61):
    position key = ((2, 0), (8, 0))
    path         = [(2, 0), (3, 0), (4, 0), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (8, 0)]
  gate (38, 10):
    position key = ((23, 2), (16, 0))
    path         = [(23, 2), (24, 2), (24, 3), (23, 3), (22, 3), (21, 3), (20, 3), (20, 2), (19, 2), (19, 1), (18, 1), (17, 1), (16, 1), (16, 0)]
  gate (53, 62):
    position key = ((9, 4), (24, 4))
    path         = [(9, 4), (10, 4), (10, 5), (11, 5), (12, 5), (13, 5), (14, 5), (15, 5), (16, 5), (17, 5), (18, 5), (19, 5), (20, 5), (21, 5), (22, 5), (23, 5), (24, 5), (24, 4)]
  gate (34, 43):
    position key = ((14, 1), (32, 2))
    path         = [(14, 1), (14, 0), (13, 0), (13, -1), (14, -1), (15, -1), (16, -1), (17, -1), (18, -1), (19, -1), (20, -1), (21, -1), (22, -1), (23, -1), (24, -1), (

In [30]:
print_vdp_layers_with_qubit_labels([schedule[10]["vdp_dict"]], schedule[10]["layout"])


Layer 0
----------------------------------------
  gate (9, 95):
    position key = ((15, 0), (40, 6))
    path         = [(15, 0), (15, -1), (16, -1), (17, -1), (18, -1), (19, -1), (20, -1), (21, -1), (22, -1), (23, -1), (23, 0), (24, 0), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (29, 2), (30, 2), (30, 3), (31, 3), (32, 3), (33, 3), (34, 3), (35, 3), (35, 4), (36, 4), (36, 5), (37, 5), (38, 5), (39, 5), (40, 5), (41, 5), (41, 6), (40, 6)]
